# Notebook 03 — Per-Horizon Model Training
## AeroTwinML · Separate Models for 24h, 48h, 72h

**Objective:** Train and evaluate multiple ML models for each forecast horizon independently.

**Why per-horizon?**
- 24h forecasts are easier (more autocorrelation) than 72h
- Different model types may win at different horizons
- Each horizon has a different signal-to-noise ratio

**Models tested per horizon:**
- Persistence (baseline: repeat last value)
- Seasonal Naive (baseline: value from 24h ago)
- Ridge Regression
- Random Forest
- Gradient Boosting
- XGBoost (if installed)
- LightGBM (if installed)

**Validation:** Walk-forward (time-aware) split — 80% train, 20% test

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils.config import get
from utils.storage import load_parquet
from feature_store.feature_builder import FeatureBuilder
from models.trainer import (
    build_models_for_horizons,
    find_best_model,
    find_best_models_per_horizon,
)

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

# Load and build features
DATA_DIR = Path(get('storage.data_dir', '../data'))
df = load_parquet(DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])

builder = FeatureBuilder(df)
featured = builder.build_all()
train_df = builder.get_training_data()

print(f'Featured: {len(featured)} rows, {len(featured.columns)} columns')
print(f'Training rows (with targets): {len(train_df)}')
print(f'Cities in training data: {train_df["city"].unique().tolist() if "city" in train_df.columns else "single"}')

## 1. Prepare Feature Columns and Targets

In [ ]:
# Define feature columns (exclude metadata and targets)
exclude = ('timestamp', 'source', 'station_name', 'city', 'country',
           'dominant_pollutant', 'merged_at', 'fetched_at', 'latitude', 'longitude')
feature_cols = [
    c for c in featured.columns
    if not c.startswith('target_')
    and c not in exclude
    and featured[c].dtype in ('float64', 'float32', 'int64', 'int32')
]

# Target columns for each horizon
target_cols = {
    '24h': 'target_aqi_24h',
    '48h': 'target_aqi_48h',
    '72h': 'target_aqi_72h',
}

print(f'Features: {len(feature_cols)}')
print(f'Targets: {list(target_cols.values())}')
print(f'\nFirst 10 features: {feature_cols[:10]}')

## 2. Walk-Forward Train/Test Split

In [ ]:
# Time-based split (80/20)
split_idx = int(len(train_df) * 0.8)
train_split = train_df.iloc[:split_idx]
test_split = train_df.iloc[split_idx:]

print(f'Train: {len(train_split)} rows ({train_split["timestamp"].min()} to {train_split["timestamp"].max()})')
print(f'Test:  {len(test_split)} rows ({test_split["timestamp"].min()} to {test_split["timestamp"].max()})')

# Visualize split
fig, ax = plt.subplots(figsize=(16, 4))
aqi_col = 'aqi' if 'aqi' in train_df.columns else 'om_forecast_aqi'
if aqi_col in train_df.columns:
    ax.plot(train_split['timestamp'], train_split[aqi_col], color='#00b4d8', label='Train', linewidth=0.5)
    ax.plot(test_split['timestamp'], test_split[aqi_col], color='#ff7e00', label='Test', linewidth=0.5)
    ax.axvline(x=train_split['timestamp'].iloc[-1], color='red', linestyle='--', label='Split')
    ax.set_title('Train/Test Split')
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.show()

## 3. Train Models for All Horizons

In [ ]:
results = build_models_for_horizons(feature_cols, target_cols, train_split, test_split)

print(f'Trained {sum(len(v) for v in results.values())} models across {len(results)} horizons')
for horizon, models in results.items():
    print(f'\n{horizon}: {len(models)} models')
    for mname, mresult in models.items():
        metrics = mresult['metrics']
        rmse_key = f'rmse_{horizon}'
        mae_key = f'mae_{horizon}'
        r2_key = f'r2_{horizon}'
        rmse = metrics.get(rmse_key, 'N/A')
        mae = metrics.get(mae_key, 'N/A')
        r2 = metrics.get(r2_key, 'N/A')
        print(f'  {mname:25s} RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.3f}' if isinstance(rmse, float) else f'  {mname}: {metrics}')

## 4. Best Model Per Horizon

In [ ]:
best_per_horizon = find_best_models_per_horizon(results)

print('Best model per horizon:')
print('=' * 60)
for horizon, entry in best_per_horizon.items():
    print(f'  {horizon}: {entry["model_name"]:20s} RMSE={entry["rmse"]:.3f}')

# Also show overall best (for comparison)
best_mname, best_horizon, best_model, all_metrics = find_best_model(results)
print(f'\nOverall best: {best_mname} @ {best_horizon}')

## 5. Model Comparison Visualization

In [ ]:
# Build comparison table
comparison = []
for horizon, models in results.items():
    for mname, mresult in models.items():
        metrics = mresult['metrics']
        comparison.append({
            'horizon': horizon,
            'model': mname,
            'rmse': metrics.get(f'rmse_{horizon}', np.nan),
            'mae': metrics.get(f'mae_{horizon}', np.nan),
            'r2': metrics.get(f'r2_{horizon}', np.nan),
        })

comp_df = pd.DataFrame(comparison)

# RMSE comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric in zip(axes, ['rmse', 'mae', 'r2']):
    pivot = comp_df.pivot(index='model', columns='horizon', values=metric)
    pivot.plot(kind='bar', ax=ax, rot=45)
    ax.set_title(f'{metric.upper()} by Model and Horizon')
    ax.grid(True, alpha=0.2)
    ax.legend(title='Horizon')

plt.tight_layout()
plt.show()

# Full comparison table
print('\nFull Comparison Table:')
pivot_rmse = comp_df.pivot(index='model', columns='horizon', values='rmse').round(3)
display(pivot_rmse)

## 6. Per-Horizon Model Persistence

In [ ]:
from models.registry import save_models_by_horizon

# Save per-horizon models
save_models_by_horizon(best_per_horizon)

print('Saved per-horizon models:')
for horizon, entry in best_per_horizon.items():
    path = DATA_DIR.parent / 'models' / 'artifacts' / f'aqi_forecaster_{horizon}.pkl'
    print(f'  {horizon}: {entry["model_name"]} -> {path}')

## Summary

**Key results:**
- Each horizon has its own best model (may differ by horizon)
- 24h typically has lowest RMSE (most predictable)
- 72h has highest RMSE (hardest to predict)
- ML models should beat baselines (persistence, seasonal naive)
- Per-horizon models saved for inference

**Next:** Notebook 04 — SHAP Explainability